<a href="https://colab.research.google.com/github/AbdullahAlNoman2144/Abdullah-portfolio/blob/main/Trying_SpaceRecognize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets torchaudio librosa jiwer
!pip install -q git+https://github.com/huggingface/peft.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import pandas as pd
import torchaudio
import librosa
import os
from datasets import Dataset, DatasetDict, Audio
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, TrainingArguments, Trainer
import torch
import random

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/Thesis Task/Datasets"

In [ ]:
print(os.listdir(BASE_DIR))

['Data P.zip', 'train.csv.zip', 'f_dataset.csv']


In [ ]:
import zipfile

In [ ]:

# Unzip Data P.zip
with zipfile.ZipFile(os.path.join(BASE_DIR, "Data P.zip"), 'r') as zip_ref:
    zip_ref.extractall("/content/audio")

In [ ]:
CSV_PATH = "/content/drive/MyDrive/Thesis Task/Datasets/f_dataset.csv"
AUDIO_PATH = "/content/audio/Data P"

In [ ]:
print(os.listdir(AUDIO_PATH))

['0a57a4293266.mp3', '1b33a5627ff8.mp3', '0ae21047256c.mp3', '1a17b8e14464.mp3', '1b07e17f0907.mp3', '1d1f9e0e428f.mp3', '1dc3f2147bc3.mp3', '0a8339b7c050.mp3', '1d0c447be06c.mp3', '1b2ceb477528.mp3', '1c0d28b6ee46.mp3', '01f75775bdfc.mp3', '1d887be96abf.mp3', '0a2adfe731f6.mp3', '1ca52e563393.mp3', '1bd18c0bdf0f.mp3', '1d4c341bcee0.mp3', '0a9452aa1856.mp3', '1bb023590f6d.mp3', '1ac11abd5952.mp3', '1bb5418c4a57.mp3', '0a82bc3f9934.mp3', '0a59b26761c5.mp3', '1d8d881b96fd.mp3', '1b81bcf6c60c.mp3', '01a0c85841e2.mp3', '1dbc0affec15.mp3', '01ad367b884c.mp3', '0b04e194bfe1.mp3', '1a7c8c463d3e.mp3', '0a392956dd70.mp3', '0a5f40308ff0.mp3', '0b4b822e3764.mp3', '1b0866ff2c86.mp3', '1cc51e2362a6.mp3', '0a87912c9c4f.mp3', '0a89cc3d4471.mp3', '0aeb5c04b285.mp3', '1a53c77fb640.mp3', '0abc88ed87e4.mp3', '1c45f6ccc218.mp3', '1d5ee6456dea.mp3', '1db8cf93c277.mp3', '1b2b4b87c074.mp3', '01d76d1cb3bf.mp3', '1c2a3a6f344a.mp3', '1d74a653d17c.mp3', '1b2cfa81d3f1.mp3', '01ae65cd7ff3.mp3', '0a5831ab7868.mp3',

In [ ]:
df = pd.read_csv(CSV_PATH)
df['path'] = df['id'].apply(lambda x: os.path.join(AUDIO_PATH, x + '.mp3'))

# Split into train and validation
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'valid'].reset_index(drop=True)

# Convert to HF dataset
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

dataset = DatasetDict({
    "train": train_ds,
    "validation": val_ds
})

In [ ]:
dataset = dataset.rename_column("sentence", "text")

In [ ]:
from datasets import Audio
dataset = dataset.cast_column("path", Audio(sampling_rate=16000))

In [ ]:
from datasets import Audio
dataset = dataset.cast_column("path", Audio(sampling_rate=16000))

def prepare_dataset(batch):
    if isinstance(batch["path"], dict) and "array" in batch["path"]:
        audio_array = batch["path"]["array"]
    else:
        raise ValueError("Audio not properly cast. Check 'cast_column'.")

    inputs = processor(audio_array, sampling_rate=16000, return_attention_mask=True)

    batch["input_values"] = inputs.input_values[0]
    batch["attention_mask"] = inputs.attention_mask[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch


In [ ]:
vocab_set = set("".join(df["sentence"].tolist()))
vocab_list = sorted(list(vocab_set))
vocab_dict = {v: k for k, v in enumerate(vocab_list)}
vocab_dict["|"] = vocab_dict[" "]  # word delimiter
vocab_dict["[PAD]"] = len(vocab_dict)  # blank token for CTC

# Save vocab
import json
with open("vocab.json", "w") as f:
    json.dump(vocab_dict, f)


from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",             # used as CTC blank
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)


In [ ]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer)
)

model.freeze_feature_encoder()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.77k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-xlsr-53 and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
print("Vocab size:", len(processor.tokenizer))
print("Pad token ID (blank):", processor.tokenizer.pad_token_id)
print("Sample:", processor.tokenizer("আমি ভালো আছি").input_ids)

Vocab size: 85
Pad token ID (blank): 81
Sample: [16, 50, 60, 0, 49, 59, 53, 67, 0, 16, 32, 60]


In [ ]:
from datasets import load_from_disk
dataset = load_from_disk("file:///content/drive/MyDrive/wav2vec_bangla_preprocessed3")

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


In [ ]:
from jiwer import wer

def compute_metrics(pred):
    pred_ids = torch.argmax(torch.tensor(pred.predictions), dim=-1)
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    return {"wer": wer(label_str, pred_str)}

In [ ]:
from transformers import TrainingArguments

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/wav2vec2-bangla-model",
    per_device_train_batch_size=8,
    eval_strategy="steps",
    num_train_epochs=5,
    fp16=True,
    save_steps=500,
    eval_steps=500,
    logging_steps=100,
    learning_rate=1e-4,
    warmup_steps=500,
    save_total_limit=2,
    gradient_accumulation_steps=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=lambda data: {
        'input_values': torch.tensor([f["input_values"] for f in data], dtype=torch.float32),
        'attention_mask': torch.tensor([f["attention_mask"] for f in data], dtype=torch.long),
        'labels': torch.tensor([f["labels"] for f in data], dtype=torch.long)
    },
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor,
    compute_metrics=compute_metrics,
)

/tmp/ipython-input-22-807359429.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./wav2vec2-fine-tuned",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    eval_strategy="steps",  # Corrected parameter name
    num_train_epochs=10,
    gradient_checkpointing=True,
    fp16=True,
    save_steps=100,
    eval_steps=100,
    logging_steps=100,
    learning_rate=1e-4,
    weight_decay=0.005,
    warmup_steps=1000,
    save_total_limit=2,
)

In [ ]:
from transformers import Trainer
import torch

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=lambda data: {
        'input_values': torch.nn.utils.rnn.pad_sequence([torch.tensor(f["input_values"], dtype=torch.float32) for f in data], batch_first=True, padding_value=0.0),
        'attention_mask': torch.nn.utils.rnn.pad_sequence([torch.tensor(f["attention_mask"], dtype=torch.long) for f in data], batch_first=True, padding_value=0),
        'labels': torch.nn.utils.rnn.pad_sequence([torch.tensor(f["labels"], dtype=torch.long) for f in data], batch_first=True, padding_value=processor.tokenizer.pad_token_id)
    },
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor.tokenizer,
)

/tmp/ipython-input-24-1867800355.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Now you can start the training by running:

In [ ]:
# Check for characters in the dataset that are not in the vocabulary
all_chars = set("".join(df["sentence"].tolist()))
chars_not_in_vocab = all_chars - vocab_set

if chars_not_in_vocab:
    print(f"Characters in the dataset not in the vocabulary: {chars_not_in_vocab}")
else:
    print("All characters in the dataset are in the vocabulary.")